# AgriEngineering 4664292 · NB16 — Ensayos de revisión metodológica
**Manuscrito:** *Geometric Invariance and Representation Sufficiency in Morphometric Bean and Rice Classification*  
**Cuestiones:** Revisor 1 comentario 4 · Revisor 2 comentarios 2, 3, 4, 5, 6, 7 · procedencia Rice 2025/2026.

**Hardware requerido:** ✅ **CPU / Entorno de ejecución → Cambiar tipo de entorno de ejecución → CPU**. No utilizar GPU; el notebook **lee predicciones congeladas** y realiza análisis estadísticos con NumPy, pandas y SciPy. No entrena nuevamente R1/R3/R4 y evita mezclar el ajuste confirmatorio con el modelo campeón original de Notebook 05.

**Orden:** 1) configurar Drive 2) ejecutar todas las celdas en secuencia 3) revisar CSV/PNG del directorio de salida en Drive 4) enviarnos el resumen y el ZIP de resultados. **NO actualizar todavía GitHub ni introducir conclusiones en el manuscrito.**

**Protección metodológica:** no se modifican `data/*_frozen_v1.csv`, `splits/*_manifest_v1.csv`, ni las predicciones del Notebook 12. Las nuevas pruebas son análisis de revisión, NO resultados previamente especificados. Cada archivo se guarda con la etiqueta `NB16`.

## 0. Instalación y librerías
Se utiliza Python 3 en Colab. `pyarrow` permite leer los archivos Parquet del ensayo confirmatorio; se instala sólo si hace falta. **CPU** es suficiente.

In [ ]:
import sys, subprocess, importlib.util
needed = [('pyarrow','pyarrow'),('pandas','pandas'),('scipy','scipy'),('scikit-learn','sklearn'),('matplotlib','matplotlib')]
missing=[pkg for pkg,mod in needed if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from pathlib import Path
import os, json, shutil, zipfile, hashlib, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from scipy.stats import binomtest, ks_2samp
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, accuracy_score,
                             precision_recall_fscore_support, log_loss)
from sklearn.model_selection import train_test_split
from IPython.display import display
SEED = 20261027
RNG = np.random.default_rng(SEED)
N_PERMUTATIONS = 20000  # Comparación por grupos (Monte Carlo, no es el p de McNemar fila a fila)
N_BOOT = 2000
print('Python:',sys.version.split()[0], '| CPU reproducible | N_BOOT:',N_BOOT)

## 1. Montar Google Drive e identificar las fuentes
El repositorio antiguo está en `Mi unidad/Grain_Robustness_Q1`. El archivo ZIP de Notebook 12 ya está guardado allí. Este notebook trabaja sobre **una copia local temporal del ZIP**, mientras exporta las tablas finales al directorio `Revision_4664292_Ensayos_2026/resultados_colab`.

El archivo complementario **`INIAP_2025_2026_campaign_manifest_record_id_v1.csv`** identifica cada registro original de 2025 y cada registro adicional de 2026. Se adjunta junto al notebook para no volver a reconstruir las campañas manualmente.

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
MYDRIVE=Path('/content/drive/MyDrive')
PROJECT=MYDRIVE/'Grain_Robustness_Q1'
REVIEW_DIR=PROJECT/'Revision_4664292_Ensayos_2026'
if not PROJECT.is_dir():
    raise FileNotFoundError('No existe Mi unidad/Grain_Robustness_Q1. Verifique la cuenta Google conectada a Colab.')
REVIEW_DIR.mkdir(exist_ok=True)
OUT=REVIEW_DIR/'resultados_colab'; OUT.mkdir(parents=True,exist_ok=True)
ZIP_NAME='NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2.zip'
zip_candidates=list(PROJECT.rglob(ZIP_NAME))
if len(zip_candidates)!=1:
    raise FileNotFoundError(f'No se localizó un ZIP único {ZIP_NAME} en {PROJECT}; coincidencias: {zip_candidates}')
zip_src=zip_candidates[0]
manifest_path=REVIEW_DIR/'INIAP_2025_2026_campaign_manifest_record_id_v1.csv'
if not manifest_path.exists():
    raise FileNotFoundError('Falta el CSV de procedencia junto al notebook en la carpeta Revision_4664292_Ensayos_2026')
work=Path('/content/nb16_agriengineering'); work.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zip_src) as archive:
    # Nunca sobreescribimos el ZIP fuente ni las tablas del repositorio.
    archive.extractall(work)
print('ZIP fuente:',zip_src,'| SHA256:', hashlib.sha256(zip_src.read_bytes()).hexdigest())
print('Manifest:', manifest_path,'| SHA256:', hashlib.sha256(manifest_path.read_bytes()).hexdigest())
print('Resultados a:',OUT)

## 2. Recuperación y alineación estricta de las predicciones del Notebook 12
Para cada representación se exige un registro único por `record_id`, coincidencia de etiqueta verdadera y grupo de duplicado, y alineación exacta de los mismos individuos del test. Los resultados confirmatorios de control deben reproducirse sin reentrenamiento. **Si una cifra no cuadra, el notebook se detiene.**

In [ ]:
LABELS={
    'dry_bean':{'R1':'R1_full_native','R3-C':'R3_complete_invariant','R4-C':'R4_complete_log_area'},
    'iniap_rice':{'R1':'R1_full_native','R3-NR':'R3_complete_invariant','R4-NR':'R4_complete_log_area'}
}
CHECK={('dry_bean','R1'):0.935439598153466,('dry_bean','R3-C'):0.7620670285392909,
       ('dry_bean','R4-C'):0.9330283682823068,('iniap_rice','R1'):0.977586916037337,
       ('iniap_rice','R3-NR'):0.97814222501307,('iniap_rice','R4-NR'):0.9811650961412388}
def load_preds(dataset,rep,partition='test'):
    target=LABELS[dataset][rep]
    opts=[p for p in work.rglob(f'rbf_svm_{partition}_predictions.*')
          if p.parent.name==target and p.parent.parent.name==dataset and p.suffix in ('.parquet','.csv')]
    opts.sort(key=lambda p:p.suffix!='.parquet')
    if not opts:
        raise FileNotFoundError(f'No se encontró la predicción {dataset}/{target}/{partition}. Revisar contenido de ZIP NB12.')
    file=opts[0]
    data=pd.read_parquet(file) if file.suffix=='.parquet' else pd.read_csv(file)
    mandatory={'record_id','duplicate_group_id','class_label','predicted_native'}
    if not mandatory.issubset(data.columns): raise ValueError(f'Columnas faltantes {mandatory-set(data.columns)} en {file}')
    data=data.sort_values('record_id').reset_index(drop=True)
    if data['record_id'].duplicated().any():raise AssertionError(f'record_id duplicado: {file}')
    for col in ['record_id','duplicate_group_id','class_label','predicted_native']:
        data[col]=data[col].astype(str)
    return data
PRED={}; METRICS=[]
for dataset,reps in LABELS.items():
    baseline=None
    for name in reps:
        frame=load_preds(dataset,name)
        keys=frame[['record_id','class_label','duplicate_group_id']]
        if baseline is None: baseline=keys
        elif not baseline.equals(keys): raise AssertionError('Falla de alineación entre representaciones '+dataset)
        macro=f1_score(frame.class_label,frame.predicted_native,average='macro')
        expected=CHECK[(dataset,name)]
        if abs(macro-expected)>0.00005:
            raise AssertionError(f'{dataset}/{name} no reproduce Notebook12: {macro} vs {expected}')
        PRED[(dataset,name)]=frame
        METRICS.append({'dataset':dataset,'representation':name,'n_test':len(frame),'macro_f1':macro})
controls=pd.DataFrame(METRICS); display(controls)
controls.to_csv(OUT/'NB16_00_verificacion_metricas_confirmatorias.csv',index=False)

## 3. Comentario 4 del primer revisor — F1 por clase y matrices comparativas
**Fréjol:** R1, R3-C, R4-C (mismo refit confirmatorio).  **Arroz:** R1 y R3-NR (además R4-NR como sensibilidad). Se producirán CSV y seis PNG. Se describe especialmente la confusión **SIRA↔DERMASON** sin atribuir diferencias a tamaños no verificados.

In [ ]:
CLASS_ROWS=[];CONF_ROWS=[]
for dataset,reps in LABELS.items():
    for name in reps:
        fr=PRED[(dataset,name)]
        labels=sorted(fr['class_label'].unique())
        r=classification_report(fr.class_label,fr.predicted_native,labels=labels,output_dict=True,zero_division=0)
        for label in labels:
            CLASS_ROWS.append({'dataset':dataset,'representation':name,'class':label,**r[label]})
        mat=confusion_matrix(fr.class_label,fr.predicted_native,labels=labels)
        for i,true in enumerate(labels):
            for j,pred in enumerate(labels):
                CONF_ROWS.append({'dataset':dataset,'representation':name,'true_class':true,'predicted_class':pred,'count':int(mat[i,j])})
        fig,ax=plt.subplots(figsize=(max(6,1.06*len(labels)), max(5,0.9*len(labels))))
        img=ax.imshow(mat,cmap='Blues')
        ax.set_xticks(range(len(labels)),labels=labels,rotation=40,ha='right')
        ax.set_yticks(range(len(labels)),labels=labels)
        ax.set(xlabel='Predicción',ylabel='Clase verdadera',title=f'{dataset}: {name} — test confirmatorio')
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j,i,str(mat[i,j]),ha='center',va='center',fontsize=9)
        fig.colorbar(img,ax=ax,fraction=.048,pad=.03)
        fig.tight_layout(); fig.savefig(OUT/f'NB16_confusion_{dataset}_{name}.png',dpi=200);plt.close(fig)
class_table=pd.DataFrame(CLASS_ROWS);conf_table=pd.DataFrame(CONF_ROWS)
class_table.to_csv(OUT/'NB16_01_f1_por_clase.csv',index=False)
conf_table.to_csv(OUT/'NB16_02_matrices_confusion_long.csv',index=False)
display(class_table[['dataset','representation','class','f1-score','support']])
bean=conf_table[conf_table.dataset=='dry_bean']
display(bean[bean.true_class.isin(['SIRA','DERMASON']) & bean.predicted_class.isin(['SIRA','DERMASON'])].sort_values(['representation','true_class','predicted_class']))

## 4. Comentario 5 del segundo revisor — inferencia que respeta duplicados
El McNemar fila a fila del texto es **diagnóstico histórico**: 68 parejas de filas idénticas en Dry Bean no son unidades independientes. Aquí evaluamos el contraste con dos alternativas:

1. **Prueba de intercambio de modelos por clúster (`duplicate_group_id`)**: se invierte el signo de toda la contribución de un grupo conjuntamente. Reportamos el p bilateral Monte Carlo con corrección `+1`; no es el p exacto fila a fila y no debe llamarse así.
2. **McNemar exacto sobre una fila representativa determinista por grupo** (fila de menor `record_id`): prueba de sensibilidad conservadora; descarta réplicas, no reemplaza la prueba de permutación.

Además, bootstrap pareado por grupos dentro de cada clase para IC del cambio de macro-F1. Para controlar multiplicidad, Holm por cultivo. No tratamos el ajuste sobre test como optimización.

In [ ]:
from statsmodels.stats.multitest import multipletests
COMPARE={'dry_bean':[('R1','R3-C'),('R1','R4-C')],
         'iniap_rice':[('R1','R3-NR'),('R1','R4-NR')]}

def bootstrap_delta_groups(mrg, A, B, n=2000,seed=SEED):
    rng=np.random.default_rng(seed)
    class_order=sorted(mrg['class_label'].unique())
    grouped={}
    for cl,sub in mrg.groupby('class_label',sort=True):
        grouped[cl]=[g.index.to_numpy() for _,g in sub.groupby('duplicate_group_id',sort=True)]
    y=mrg.class_label.to_numpy();pa=mrg[A].to_numpy();pb=mrg[B].to_numpy()
    out=np.empty(n)
    for i in range(n):
        indices=[]
        for cl in class_order:
            groups=grouped[cl]
            choices=rng.integers(0,len(groups),size=len(groups))
            indices.extend(np.concatenate([groups[z] for z in choices]).tolist())
        inds=np.asarray(indices,dtype=int)
        out[i]=f1_score(y[inds],pa[inds],labels=class_order,average='macro',zero_division=0)-f1_score(y[inds],pb[inds],labels=class_order,average='macro',zero_division=0)
    return np.percentile(out,[2.5,97.5]),np.percentile(out,[5,95])

def cluster_swap_p(mrg, a,b,n=20000,seed=SEED):
    z=(mrg[a].eq(mrg.class_label).astype(int)-mrg[b].eq(mrg.class_label).astype(int))
    v=z.groupby(mrg.duplicate_group_id,sort=True).sum().to_numpy(float)
    obs=abs(v.sum());rng=np.random.default_rng(seed);extreme=0
    for start in range(0,n,500):
        k=min(500,n-start)
        signs=rng.choice(np.array([-1.,1.]),size=(k,len(v)))
        extreme+=np.count_nonzero(abs(signs@v)>=obs-1e-12)
    return (extreme+1)/(n+1),len(v),obs

infer=[]
for dataset,pairs in COMPARE.items():
    for A,B in pairs:
        a=PRED[(dataset,A)][['record_id','duplicate_group_id','class_label','predicted_native']].rename(columns={'predicted_native':'p_A'})
        b=PRED[(dataset,B)][['record_id','predicted_native']].rename(columns={'predicted_native':'p_B'})
        dat=a.merge(b,on='record_id',validate='one_to_one').reset_index(drop=True)
        corr_A=dat.p_A.eq(dat.class_label);corr_B=dat.p_B.eq(dat.class_label)
        discord_b=int((corr_A & ~corr_B).sum());discord_c=int((~corr_A & corr_B).sum())
        # una fila por clúster, con orden determinista de record_id
        g=dat.sort_values('record_id').drop_duplicates('duplicate_group_id')
        ga=g.p_A.eq(g.class_label);gb=g.p_B.eq(g.class_label)
        bi=int((ga & ~gb).sum());ci=int((~ga & gb).sum())
        p_exact=float(binomtest(min(bi,ci),bi+ci,p=.5,alternative='two-sided').pvalue) if bi+ci else 1.
        p_swap,n_groups,obs=cluster_swap_p(dat,'p_A','p_B',N_PERMUTATIONS)
        ci95,ci90=bootstrap_delta_groups(dat,'p_A','p_B',N_BOOT)
        delta=f1_score(dat.class_label,dat.p_A,average='macro')-f1_score(dat.class_label,dat.p_B,average='macro')
        infer.append({'dataset':dataset,'contrast':f'{A} − {B}','n_rows':len(dat),'n_groups':n_groups,'row_b':discord_b,'row_c':discord_c,
            'group_representative_b':bi,'group_representative_c':ci,'group_representative_exact_mcnemar_p':p_exact,
            'cluster_swap_p_mc':p_swap,'cluster_swap_abs_stat':obs,'macro_f1_delta':delta,
            'ci95_low':ci95[0],'ci95_high':ci95[1],'ci90_low':ci90[0],'ci90_high':ci90[1],
            'equiv_001_exploratory':bool(ci90[0]>-.01 and ci90[1]<.01),'equiv_0005_exploratory':bool(ci90[0]>-.005 and ci90[1]<.005)})
infer=pd.DataFrame(infer)
for crop,ids in infer.groupby('dataset').groups.items():
    for col in ['cluster_swap_p_mc','group_representative_exact_mcnemar_p']:
        infer.loc[ids,col+'_holm']=multipletests(infer.loc[ids,col],method='holm')[1]
infer.to_csv(OUT/'NB16_03_McNemar_agrupado_y_bootstrap.csv',index=False)
display(infer[['dataset','contrast','row_b','row_c','n_groups','cluster_swap_p_mc','group_representative_exact_mcnemar_p','macro_f1_delta','ci95_low','ci95_high']])
print('ATENCIÓN: los IC a ±0.01 son análisis de sensibilidad, no justifican agronómicamente un margen.')

## 5. Comentarios 4 y 7 del segundo revisor — calibración independiente, APS y carga por variedad
**Sin reentrenar clasificadores**. Se usa el conjunto `calibration` del Notebook 12 pero se divide, **a nivel de `duplicate_group_id` y estratificado por clase**, en tres subgrupos independientes: 40% para estimar temperatura, 30% para fijar el umbral de abstención y 30% para cuantiles conformales. El test queda intacto. Esto elimina la reutilización de las mismas observaciones para las tres operaciones.

Se presentan 90% y 95% APS aleatorizado con semilla fija; clase individual, tamaño y vacíos. Las tasas son **empíricas** y no se prometen garantías condicionales por clase. La validez marginal requiere intercambiabilidad y un mecanismo de randomización apropiado. Se informa además un test de clasificación selectiva con punto operativo 90% calibrado sin mirar test.

In [ ]:
from scipy.special import softmax

def get_proba(frame, prefix='prob_native_'):
    columns=sorted((x for x in frame.columns if x.startswith(prefix)),key=lambda s:int(s.rsplit('_',1)[-1]))
    cls=[]
    for x in columns:
        i=int(x.rsplit('_',1)[-1]); cname=f'class_{i}'
        if cname not in frame: raise KeyError(f'Falta {cname}')
        assert frame[cname].nunique()==1
        cls.append(str(frame[cname].iloc[0]))
    proba=frame[columns].to_numpy(float)
    assert len(columns)>=2 and np.allclose(proba.sum(axis=1),1,atol=1e-5)
    return proba,np.array(cls,dtype=str)

def tscale(p,T):
    return softmax(np.log(np.clip(p,1e-12,1))/T,axis=1)

def optimize_T(P,y,classes):
    fun=lambda logT:log_loss(y,tscale(P,np.exp(logT)),labels=classes)
    r=minimize_scalar(fun,bounds=(-3,3),method='bounded')
    if not r.success:raise RuntimeError('Fallo optimización temperatura')
    return float(np.exp(r.x))

def cal_partitions(df):
    # agrupamiento por clase; no se divide un grupo entre particiones
    groups=df[['duplicate_group_id','class_label']].drop_duplicates()
    if df.groupby('duplicate_group_id').class_label.nunique().gt(1).any():
        raise AssertionError('Hay grupos de duplicado con más de una clase')
    ids=groups.duplicate_group_id.to_numpy()
    yy=groups.class_label.to_numpy()
    tr,rem=train_test_split(np.arange(len(groups)),train_size=.40,random_state=SEED,stratify=yy)
    sel,conf=train_test_split(rem,train_size=.50,random_state=SEED+1,stratify=yy[rem])
    parts={'temperature':set(ids[tr]),'selective':set(ids[sel]),'conformal':set(ids[conf])}
    assert not(parts['temperature']&parts['selective'] or parts['temperature']&parts['conformal'] or parts['selective']&parts['conformal'])
    return {k:np.where(df.duplicate_group_id.isin(v).to_numpy())[0] for k,v in parts.items()}

def aps_scores(P,y,classes,U):
    idx={cl:i for i,cl in enumerate(classes)}
    cand=np.zeros_like(P)
    for i,row in enumerate(P):
        order=np.argsort(-row,kind='stable');cumsum=np.cumsum(row[order]);before=np.r_[0.,cumsum[:-1]]
        cand[i,order]=before+U[i]*row[order]
    return cand, np.array([cand[i,idx[label]] for i,label in enumerate(y)])

def quantile_higher(scores,alpha):
    n=len(scores);j=int(np.ceil((n+1)*(1-alpha)))
    return float(np.sort(scores)[j-1]) if j<=n else float('inf')

def randomized_group_u(group_ids,seed=SEED+100):
    # orden estable, una U por grupo: sin reutilizar grupo en calibración ni test
    groups=sorted(set(group_ids)); rng=np.random.default_rng(seed)
    dd=dict(zip(groups,rng.random(len(groups))))
    return np.array([dd[v] for v in group_ids],float)

unc_rows=[];group_rows=[];sel_rows=[];class_rows=[]
for dataset,champ in [('dry_bean','R1'),('iniap_rice','R4-NR')]:
    cal=load_preds(dataset,champ,'calibration')
    test=PRED[(dataset,champ)]
    PC,classes=get_proba(cal); PT,cl_test=get_proba(test)
    assert np.array_equal(classes,cl_test)
    split=cal_partitions(cal)
    temperature=optimize_T(PC[split['temperature']],cal.class_label.to_numpy()[split['temperature']],classes)
    Pcal=tscale(PC,temperature);Ptest=tscale(PT,temperature)
    yy=cal.class_label.to_numpy();yt=test.class_label.to_numpy()
    # clasificación selectiva: umbral fijado en subgrupo exclusivamente selectivo
    cert=Pcal.max(axis=1);test_cert=Ptest.max(axis=1)
    threshold=float(np.quantile(cert[split['selective']],0.10))
    accept=test_cert>=threshold
    selected_acc=float(accuracy_score(yt[accept],classes[Ptest.argmax(axis=1)][accept])) if accept.any() else np.nan
    sel_rows.append({'dataset':dataset,'representation':champ,'temperature':temperature,'n_temperature':len(split['temperature']),
        'n_selective_calibration':len(split['selective']),'n_conformal_calibration':len(split['conformal']),
        'certainty_threshold':threshold,'test_coverage':float(accept.mean()),'test_selected_accuracy':selected_acc})
    for cl in classes:
        mask=(yt==cl);class_rows.append({'dataset':dataset,'class':cl,'test_support':int(mask.sum()),
            'review_fraction':float((~accept[mask]).mean()),'retained_fraction':float(accept[mask].mean())})
    # grupo-level calibración APS: una fila representativa por grupo de calibración conforme
    conf_indices=split['conformal'];groups_cal=cal.iloc[conf_indices].sort_values('record_id').drop_duplicates('duplicate_group_id')
    ids_cal=groups_cal.index.to_numpy()
    group_test=test.sort_values('record_id').drop_duplicates('duplicate_group_id')
    ids_test=group_test.index.to_numpy()
    Ucal=randomized_group_u(cal.duplicate_group_id.to_numpy(),seed=SEED+110)
    Utest=randomized_group_u(test.duplicate_group_id.to_numpy(),seed=SEED+120)
    score_matrix_cal,true_cal=aps_scores(Pcal[ids_cal],yy[ids_cal],classes,Ucal[ids_cal])
    score_matrix_test,true_test=aps_scores(Ptest,yt,classes,Utest)
    for alpha in [0.10,0.05]:
        q=quantile_higher(true_cal,alpha)
        sets=score_matrix_test<=q
        truth_idx=np.array([int(np.where(classes==cl)[0][0]) for cl in yt])
        hit=sets[np.arange(len(yt)),truth_idx]
        empty=~sets.any(axis=1)
        unc_rows.append({'dataset':dataset,'method':'APS_independent_calibration','nominal_coverage':1-alpha,
            'n_conformal_groups':len(ids_cal),'q':q,'test_rows':len(yt),
            'row_coverage':float(hit.mean()),'row_empty_fraction':float(empty.mean()),'row_mean_set_size':float(sets.sum(axis=1).mean()),
            'group_representative_coverage':float(hit[ids_test].mean()),'group_representative_empty_fraction':float(empty[ids_test].mean())})
        for cl in classes:
            mask=yt==cl
            group_rows.append({'dataset':dataset,'method':'APS_independent_calibration','nominal_coverage':1-alpha,'class':cl,
                'support':int(mask.sum()),'coverage':float(hit[mask].mean()),
                'empty_fraction':float(empty[mask].mean()),'mean_set_size':float(sets.sum(axis=1)[mask].mean())})
res_sel=pd.DataFrame(sel_rows);res_unc=pd.DataFrame(unc_rows);res_cls=pd.DataFrame(group_rows);res_work=pd.DataFrame(class_rows)
res_sel.to_csv(OUT/'NB16_04_split_calibracion_selectiva.csv',index=False)
res_unc.to_csv(OUT/'NB16_05_APS_calibracion_independiente_global.csv',index=False)
res_cls.to_csv(OUT/'NB16_06_APS_por_clase.csv',index=False)
res_work.to_csv(OUT/'NB16_07_carga_revision_por_clase.csv',index=False)
display(res_sel);display(res_unc);display(res_cls);display(res_work)

## 6. Procedencia de los 2.400 + 6.800 granos INIAP y sensibilidad entre campañas
La tabla de origen corresponde a un **cotejo de identidad 2.400/2.400 entre Excel 2025 y consolidado**. Es **evidencia de procedencia tabular**, no prueba de que cada fila represente otro lote biológico. Se calcula macro-F1 por campaña en la partición de prueba y la distribución de frecuencias por clase; **no usar esta comparación para cambiar representaciones ni seleccionar modelos**. La diferencia temporal y de smartphone podría coexistir con diferencias de operador o lote.

In [ ]:
mp=pd.read_csv(manifest_path,dtype={'record_id':str,'acquisition_campaign':str})
assert mp.record_id.is_unique and len(mp)==9200
assert mp.acquisition_campaign.value_counts().to_dict()=={'2026_extension':6800,'2025_original':2400}
mi=[];class_dist=[]
for rep in ['R1','R3-NR','R4-NR']:
    t=PRED[('iniap_rice',rep)].merge(mp[['record_id','acquisition_campaign']],on='record_id',validate='one_to_one')
    if t.acquisition_campaign.isna().any():raise AssertionError('Faltan registros para campaña')
    for camp,part in t.groupby('acquisition_campaign'):
        mi.append({'representation':rep,'campaign':camp,'test_n':len(part),
          'accuracy':accuracy_score(part.class_label,part.predicted_native),
          'macro_f1':f1_score(part.class_label,part.predicted_native,average='macro')})
        for cl,count in part.class_label.value_counts().sort_index().items():
            class_dist.append({'representation':rep,'campaign':camp,'class':cl,'n_test':int(count)})
campaign=pd.DataFrame(mi); dist=pd.DataFrame(class_dist)
campaign.to_csv(OUT/'NB16_08_metricas_test_por_campana.csv',index=False)
dist.to_csv(OUT/'NB16_09_soporte_clase_por_campana.csv',index=False)
mp.to_csv(OUT/'NB16_10_manifest_procedencia_usado.csv',index=False)
display(campaign);display(dist)
print('Este es un análisis retrospectivo por campañas, NO una validación temporal independiente.')

## 7. Exportar auditoría final y ZIP de evidencias
Todos los CSV y PNG se almacenan en `Mi unidad/Grain_Robustness_Q1/Revision_4664292_Ensayos_2026/resultados_colab`.

El ZIP es apto para conservar en Drive y auditar antes de subir material derivado a GitHub. **No incluye fotografías originales, modelos propietarios, datos sin derechos de redistribución, ni archivos congelados reescritos.**

In [ ]:
metadata={'manuscript':'agriengineering-4664292','run_date_utc':pd.Timestamp.now(tz='UTC').isoformat(),
          'source_zip':str(zip_src),'source_zip_sha256':hashlib.sha256(zip_src.read_bytes()).hexdigest(),
          'campaign_manifest_sha256':hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
          'analysis_type':'post-review additional analyses; frozen NB12 predictions',
          'hardware':'CPU','bootstraps':N_BOOT,'cluster_swap_permutations':N_PERMUTATIONS,
          'python':sys.version.split()[0], 'numpy':np.__version__,'pandas':pd.__version__}
(OUT/'NB16_11_metadatos.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
checks=[]
for path in sorted(OUT.glob('NB16_*')):
    checks.append({'file':path.name,'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
pd.DataFrame(checks).to_csv(OUT/'NB16_12_SHA256_resultados.csv',index=False)
zip_out=REVIEW_DIR/'NB16_RESULTADOS_REVISORES_4664292.zip'
with zipfile.ZipFile(zip_out,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob('NB16_*')):z.write(p,arcname=p.name)
print('COMPLETADO: se han escrito',len(checks),'archivos NB16')
print('ZIP:',zip_out,'tamaño:',zip_out.stat().st_size,'bytes')
print('Cargar este ZIP aquí para revisar los resultados ANTES de editar el artículo.')

## 8. Reglas para redacción posterior (no editar el Word todavía)
1. **Poner en azul (`#0056B3`) toda incorporación o modificación textual** en el manuscrito Word. Mantener figuras y tablas actuales, añadir las nuevas sólo tras verificar los CSV.
2. Para R1 vs. R3-C y R4-C, usar F1 por clase y matrices del **mismo refit confirmatorio Notebook 12**.
3. Los resultados de la calibración partida y los p-valores por grupos son **análisis nuevos posteriores a revisión**, claramente identificados y con sus supuestos.
4. Mantener separados los resultados originales Notebook 05 (champions) de los refits de Notebook 12; registrar referencias a las tablas de procedencia.
5. Revisar además términos «size descriptor» para ECCENTRICITY, la justificación operacional de ±0.01 y la limitación de las dos campañas INIAP (no implica validación independiente de lotes).
6. No subir al GitHub las fotografías originales de Alexander; el repositorio público se limita a datos morfométricos autorizados, código y resultados auditados.